In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

RANDOM_STATE = 42

# 1. Chargement des données
DATA_PATH = "../data/Addis_Ababa_city_RTA.csv"
df = pd.read_csv(DATA_PATH)

# Normalisation légère des noms de colonnes (sans perdre l'accès aux colonnes)
df.columns = [c.strip() for c in df.columns]

# 2. Identification robuste de la colonne cible (nombre de blessés / casualties)
casualty_candidates = [
    c
    for c in df.columns
    if "casualt" in c.lower()
    and not any(
        bad in c.lower()
        for bad in ["class", "sever", "age", "sex", "fitness", "work"]
    )
]
if len(casualty_candidates) > 0:
  TARGET_COL = casualty_candidates[0]
else:
  # Repli prévu par le sujet si la colonne cible est absente du fichier
  vehicle_candidates = [c for c in df.columns if "vehicle" in c.lower() and "involved" in c.lower()]
  TARGET_COL = vehicle_candidates[0] if vehicle_candidates else df.columns[-1]

# 3. Exclusion des variables à risque de fuite de données (leakage) pour TH7 G1
leakage_cols = [
    c
    for c in df.columns
    if c != TARGET_COL
    and any(k in c.lower() for k in ["casualty", "fitness", "severity"])
]

# 4. Suppression des doublons exacts et des colonnes de fuite
df_clean = df.drop_duplicates().copy()
df_clean = df_clean.drop(columns=leakage_cols, errors="ignore")

# 5. Nettoyage des chaînes 'unknown'/vides -> np.nan et regroupement des catégories rares (< 1%)
unknown_tokens = {"unknown", "unk", "na", "n/a", "none", "null", "nan", "", "-"}
obj_cols = df_clean.select_dtypes(include=["object", "category", "string"]).columns.tolist()

for col in obj_cols:
  if col == TARGET_COL:
    continue
  s = df_clean[col].astype(str).str.strip()
  s = s.where(~s.str.lower().isin(unknown_tokens), other=np.nan)

  vc = s.value_counts(normalize=True, dropna=True)
  rare_vals = vc[vc < 0.01].index
  if len(rare_vals) > 0:
    s = s.where(~s.isin(rare_vals), other="OTHER_RARE")

  df_clean[col] = s

# Conversion propre de la cible et suppression des lignes sans cible
df_clean[TARGET_COL] = pd.to_numeric(df_clean[TARGET_COL], errors="coerce")
df_clean = df_clean.dropna(subset=[TARGET_COL]).copy()

# Suppression des éventuelles colonnes explicatives 100 % vides (all-NaN)
all_nan_cols = [c for c in df_clean.columns if c != TARGET_COL and df_clean[c].isna().all()]
if all_nan_cols:
  df_clean = df_clean.drop(columns=all_nan_cols)

y = df_clean[TARGET_COL].astype(float)
X = df_clean.drop(columns=[TARGET_COL])

# 6. Séparation des colonnes : tranches ordonnées (ordinal), nominales (one-hot) et numériques
cat_cols_clean = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

ord_cols = [
    c
    for c in cat_cols_clean
    if any(k in c.lower() for k in ["age_band", "experience", "service_year", "service_years"])
]
nom_cols = [c for c in cat_cols_clean if c not in ord_cols]

# 7. Préprocesseurs (ajustés UNIQUEMENT sur X_train à l'intérieur du Pipeline)
# Option A : Encodage mixte (Ordinal pour tranches ordonnées, One-Hot pour nominales) + StandardScaler
prep_mixed_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols,
        ),
        (
            "ord",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "ordinal",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value", unknown_value=-1
                    ),
                ),
                ("scaler", StandardScaler()),
            ]),
            ord_cols,
        ),
        (
            "nom",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                ),
            ]),
            nom_cols,
        ),
    ],
    remainder="drop",
)

# Option B : Encodage purement One-Hot sur toutes les catégorielles + StandardScaler
prep_onehot_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols,
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                ),
            ]),
            cat_cols_clean,
        ),
    ],
    remainder="drop",
)

# 8. Modèle MLPRegressor avec les paramètres fixes de la feuille T7Vr1-B6
if "PAIR_INFO" in globals() and "fixed_params" in PAIR_INFO:
  mlp_params = PAIR_INFO["fixed_params"]
else:
  mlp_params = {
      "activation": "relu",
      "solver": "adam",
      "alpha": 1e-4,
      "batch_size": 32,
      "max_iter": 500,
      "early_stopping": True,
      "random_state": RANDOM_STATE,
  }

mlp_base = MLPRegressor(**mlp_params)
pipe_mlp_mixed = Pipeline([("prep", prep_mixed_scaled), ("model", mlp_base)])
pipe_mlp_onehot = Pipeline([("prep", prep_onehot_scaled), ("model", mlp_base)])

print("=== PHASE 2 — NETTOYAGE & PIPELINES PRÊTS ===")
print(f"Cible retenue : {TARGET_COL} | Lignes conservées : {len(df_clean)}")
print(f"Features conservées : {X.shape[1]}")
print(f" - Ordinales ({len(ord_cols)}) : {ord_cols}")
print(f" - Nominales ({len(nom_cols)}) : {nom_cols}")
print(f" - Numériques ({len(num_cols)}) : {num_cols}")
print(f" - Catégorielles ({len(cat_cols_clean)}) : {cat_cols_clean}")